### Librerias

In [1]:
import pandas as pd
import openpyxl
from pathlib import Path

### Concatenacion y Procesamiento en Series

In [4]:
# ============================
# Configuración
# ============================

folder = Path(r"data\XM\Aportes")

years = range(2000, 2025)

monthly_series = []

# ============================
# Procesamiento
# ============================

for year in years:

    file = folder / f"Aportes_Mensual_{year}.xlsx"

    # print(f"Procesando {year}...")

    # Leer únicamente las columnas necesarias:
    # B = Mes
    # D = Nombre Río
    # E = Aportes Energía kWh
    df = pd.read_excel(
        file,
        header=2,
        usecols="B,D,E"
    )

    # --------------------------------
    # Limpiar nombres de columnas
    # --------------------------------

    df.columns = ["Mes", "Rio", "AportesEnergia"]

    # --------------------------------
    # Convertir aportes a numérico
    # --------------------------------

    df["AportesEnergia"] = pd.to_numeric(
        df["AportesEnergia"],
        errors="coerce"
    )

    # Eliminar filas sin datos válidos
    df = df.dropna(subset=["Mes", "AportesEnergia"])

    # --------------------------------
    # Promedio mensual entre todos los ríos
    # --------------------------------

    monthly = (
        df.groupby("Mes", as_index=False)["AportesEnergia"]
          .mean()
    )

    # --------------------------------
    # Convertir nombre del mes a número
    # --------------------------------

    meses = {
        "ENERO": 1,
        "FEBRERO": 2,
        "MARZO": 3,
        "ABRIL": 4,
        "MAYO": 5,
        "JUNIO": 6,
        "JULIO": 7,
        "AGOSTO": 8,
        "SEPTIEMBRE": 9,
        "OCTUBRE": 10,
        "NOVIEMBRE": 11,
        "DICIEMBRE": 12
    }

    monthly["Mes_num"] = (
        monthly["Mes"]
        .str.upper()
        .str.strip()
        .map(meses)
    )

    # --------------------------------
    # Crear fecha mensual
    # --------------------------------

    monthly["date"] = pd.to_datetime(
        dict(
            year=year,
            month=monthly["Mes_num"],
            day=1
        ),
        errors="coerce"
    )

    # --------------------------------
    # Conservar únicamente las columnas
    # necesarias
    # --------------------------------

    monthly = monthly[
        ["date", "AportesEnergia"]
    ]

    # --------------------------------
    # Guardar el año procesado
    # --------------------------------

    monthly_series.append(monthly)


# ============================
# Unir todos los años
# ============================

aportes_energia_monthly = (
    pd.concat(
        monthly_series,
        ignore_index=True
    )
    .sort_values("date")
    .reset_index(drop=True)
)


### Verificacion:

In [5]:
aportes_energia_monthly.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            300 non-null    datetime64[us]
 1   AportesEnergia  300 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 4.8 KB


In [6]:
aportes_energia_monthly.describe()

,date,AportesEnergia
count,300,3.000000e+02
mean,2012-06-16 02:14:24,1.641789e+08
min,2000-01-01 00:00:00,4.677111e+07
25%,2006-03-24 06:00:00,1.133542e+08
50%,2012-06-16 00:00:00,1.642640e+08
75%,2018-09-08 12:00:00,2.056831e+08
max,2024-12-01 00:00:00,3.907349e+08
std,NaN,6.458401e+07


In [8]:
aportes_energia_monthly.head()

,date,AportesEnergia
0,2000-01-01,1.055922e+08
1,2000-02-01,1.107575e+08
2,2000-03-01,1.251867e+08
3,2000-04-01,1.349806e+08
4,2000-05-01,2.676612e+08


### Guardado de las series

In [9]:
output_folder = Path(r"data\\XM\Procesadas")

output_folder.mkdir(parents=True, exist_ok=True)

aportes_energia_monthly.to_csv(
    output_folder / "aportes_energia_monthly_2000-2024.csv",
    index=False
)